# Corrigé · Exercice 2.2 : un outil réseau instable

> Notebook autonome et asynchrone. Points clés : retry **ciblé** (`ConnectionError` seulement), timeout sur des nœuds **async**, gestionnaires d'erreur qui renvoient un `Command` avec `goto`.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.errors import NodeError
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, RetryPolicy, TimeoutPolicy

from helpdesk_ops.outils import chercher_kb, statut_serveur
from helpdesk_ops.reseau import RESEAU, ping_serveur

OUTILS = [statut_serveur, chercher_kb, ping_serveur]
llm_outils = llm.bind_tools(OUTILS)
SYSTEME = SystemMessage(
    "Tu es l'agent support N1 de NordTech SA. Réponds en français, en 2 phrases maximum. "
    "Pour savoir si un serveur répond, utilise ping_serveur. Si un outil est indisponible, dis-le : n'invente jamais son résultat."
)


async def agent(state: MessagesState) -> dict:
    return {"messages": [await llm_outils.ainvoke([SYSTEME, *state["messages"]])]}


def construire(options_agent: dict | None = None, options_tools: dict | None = None):
    """Fourni : l'agent support, async. Vous ne changez QUE les options passées aux deux nœuds."""
    b = StateGraph(MessagesState)
    b.add_node("agent", agent, **(options_agent or {}))
    b.add_node("tools", ToolNode(OUTILS), **(options_tools or {}))
    b.add_edge(START, "agent")
    b.add_conditional_edges("agent", tools_condition)
    b.add_edge("tools", "agent")
    return b.compile()


QUESTION = "Le serveur srv-erp-02 répond-il au ping ?"
SCENARIOS = {
    "stable": {},
    "instable": {"taux_echec": 0.5, "graine": 3},
    "panne": {"echecs_avant_succes": 99},
    "lente": {"latence_s": 3.0},
}


async def essayer(graphe, scenario: str, session: str) -> None:
    RESEAU.configurer(**SCENARIOS[scenario])
    try:
        with propagate_attributes(session_id=session, tags=["jour-2", "ex22", scenario]):
            etat = await graphe.ainvoke({"messages": [HumanMessage(QUESTION)]},
                                        config={"callbacks": [langfuse_handler], "run_name": f"ex22-{scenario}", "recursion_limit": 10})
        print(f"✅ {scenario:<9} | {RESEAU.appels} appel(s) réseau | {etat['messages'][-1].content[:110]}")
    except Exception as erreur:
        print(f"💥 {scenario:<9} | {RESEAU.appels} appel(s) réseau | {type(erreur).__name__} : {str(erreur)[:80]}")

## 1. Le constat, sans protection

In [ ]:
INITIALES = "corrige"
fragile = construire()
for scenario in SCENARIOS:
    await essayer(fragile, scenario, f"ex22-{INITIALES}")

Les scénarios « instable » et « panne » font remonter une `ConnectionError` : l'utilisateur ne reçoit **rien**. Le scénario « lente » réussit… en 3 s par appel : acceptable ici, catastrophique si la supervision met 2 minutes.

## 2. Les protections

In [ ]:
from helpdesk_ops.config import moteur

TIMEOUT_AGENT_S = 30 if moteur() == "vllm" else 180   # salle GPU : 30 s ; poste CPU de préparation : 180 s


def outils_en_echec(state: MessagesState, error: NodeError) -> Command:
    # Une réponse par appel d'outil en attente : sinon l'historique est invalide pour le modèle
    en_attente = state["messages"][-1].tool_calls
    reponses = [ToolMessage(content=f"supervision indisponible ({type(error.error).__name__}) : résultat inconnu",
                            tool_call_id=appel["id"], name=appel["name"], status="error") for appel in en_attente]
    return Command(update={"messages": reponses}, goto="agent")   # sans goto, le graphe s'arrêterait ici


def agent_en_echec(state: MessagesState, error: NodeError) -> Command:
    print(f"⚠️  nœud {error.node} en échec, repli envoyé : {error.error!r}")   # la vraie cause reste visible
    return Command(update={"messages": [AIMessage("Je ne peux pas vous répondre pour le moment : votre demande est transmise à un technicien.")]},
                   goto=END)


OPTIONS_TOOLS = {
    "retry_policy": RetryPolicy(max_attempts=3, initial_interval=0.2, retry_on=ConnectionError),
    "timeout": TimeoutPolicy(run_timeout=1),
    "error_handler": outils_en_echec,
}
OPTIONS_AGENT = {
    "timeout": TimeoutPolicy(run_timeout=TIMEOUT_AGENT_S),
    "error_handler": agent_en_echec,
}

robuste = construire(OPTIONS_AGENT, OPTIONS_TOOLS)
for scenario in SCENARIOS:
    await essayer(robuste, scenario, f"ex22-{INITIALES}")
langfuse.flush()

## Ce qu'il fallait observer

- **stable** : 1 appel réseau, réponse normale.
- **instable** (graine 3) : des échecs absorbés par le retry ; la réponse est correcte. Dans Langfuse, les tentatives en échec apparaissent comme des observations `tools` / `ping_serveur` en **erreur**, suivies de la tentative réussie.
- **panne** : 3 tentatives, puis `outils_en_echec` ; le modèle reçoit « supervision indisponible » et le **dit** à l'utilisateur. S'il prétend malgré tout que le serveur répond, c'est une **hallucination** : à mesurer en J4.
- **lente** : chaque tentative est coupée à 1 s ; le timeout n'est **pas** réessayé (`retry_on=ConnectionError` seulement), donc repli direct. Si l'on ajoutait `NodeTimeoutError` à `retry_on`, on paierait 3 × 1 s pour rien : une supervision lente le restera dans la seconde qui suit.

## Pour aller plus loin

- Le timeout de l'agent (180 s sur CPU) n'est pas le même en salle (30 s sur GPU) : où ce réglage doit-il vivre ? *(dans la configuration, comme `MODEL_AGENT`, pas dans le code)*
- Que se passe-t-il si **deux** outils sont appelés en parallèle et qu'un seul échoue ? *(tout le nœud `tools` est réessayé : les outils doivent être idempotents)*